# Notebook 2: Greeks Analysis & Delta Hedging
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import norm
from scipy.optimize import brentq
import sys; sys.path.insert(0,'../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#0d1117',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'text.color':'#c9d1d9','grid.color':'#21262d',
    'axes.titlecolor':'#f0f6fc','legend.facecolor':'#161b22',
    'legend.edgecolor':'#30363d','font.size':11
})
COLORS = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff']
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy_ret = returns['SPY']
S0, r, sigma = 100, 0.05, 0.20
print(f'Loaded {len(spy_ret):,} observations')


Loaded 2,609 observations


In [2]:
from greeks import greeks, greeks_surface, delta_hedge_simulation
g=greeks(S0,100,1.0,r,sigma)
print("ATM Greeks (T=1yr):",[f"{k}={v:.4f}" for k,v in g.items()])

ATM Greeks (T=1yr): ['delta=0.6368', 'gamma=0.0188', 'vega=37.5240', 'theta=-6.4140', 'rho=53.2325']


In [3]:
# Greeks heatmaps
from greeks import greeks
Kg2,Tg2=np.meshgrid(np.linspace(70,135,60),np.linspace(0.05,2,60))
delta_g=np.vectorize(lambda K,T:greeks(S0,K,T,r,sigma)["delta"])(Kg2,Tg2)
gamma_g=np.vectorize(lambda K,T:greeks(S0,K,T,r,sigma)["gamma"])(Kg2,Tg2)
vega_g=np.vectorize(lambda K,T:greeks(S0,K,T,r,sigma)["vega"])(Kg2,Tg2)
theta_g=np.vectorize(lambda K,T:greeks(S0,K,T,r,sigma)["theta"])(Kg2,Tg2)
fig,axes=plt.subplots(2,2,figsize=(15,10),facecolor="#0d1117")
fig.suptitle("Options Greeks Heatmaps",color="#f0f6fc",fontsize=14,fontweight="bold")
for ax,data,name,cmap in zip(axes.flat,[delta_g,gamma_g,vega_g,theta_g],["Delta","Gamma","Vega","Theta"],["RdYlGn","hot","YlOrRd","RdBu_r"]):
    im=ax.imshow(data,aspect="auto",origin="lower",cmap=cmap,extent=[70,135,0.05,2])
    ax.axvline(S0,color="#f0f6fc",linewidth=1.5,linestyle="--",alpha=0.8)
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
    ax.set_title(name,color="#f0f6fc",fontsize=13,fontweight="bold")
    ax.tick_params(colors="#8b949e",labelsize=8)
plt.tight_layout()
plt.savefig("../results/02_greeks_heatmaps.png",dpi=150,bbox_inches="tight",facecolor="#0d1117")
plt.show()
print("Chart saved.")

Chart saved.


In [4]:
result=delta_hedge_simulation(100,100,1.0,r,sigma)
print(f"Final stock price: ${result["stock_path"][-1]:.2f}")
print(f"Total hedge P&L  : ${result["total_pnl"]:.4f}")

Final stock price: $101.82
Total hedge P&L  : $-1.9483
